# Processing Data from Parquet Files and Merging Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy as sp
import statsmodels as sm
import sklearn as sk
import os
import re
import time
from tqdm import tqdm

Potentially change working directory

In [ ]:
print(os.getcwd())

List files in directory

In [ ]:
print(os.listdir())

### Static Reference Data

Read static reference data

In [ ]:
static_reference_data_unmerged = {}
for batch_index in [1,2]:
    static_reference_data_unmerged_filenames = os.listdir(f"palate_data_parquet/batch_{batch_index}")
    for filename in tqdm(static_reference_data_unmerged_filenames):
        if ".parquet" in filename:
            df = pd.read_parquet(f"palate_data_parquet/batch_{batch_index}/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            df.columns.name = base_name
            static_reference_data_unmerged[f"{base_name}"] = df

Separate to merge

In [ ]:
tagged_data = []
static_reference_data_unmerged_without_tagged_1 = []
static_reference_data_unmerged_without_tagged_2 = []
for name, df in static_reference_data_unmerged.items():
    if "tagged" in name:
        tagged_data.append(df)
    elif "1" in name:
        df['batch'] = 1
        static_reference_data_unmerged_without_tagged_1.append(df)
    else:
        df['batch'] = 2
        static_reference_data_unmerged_without_tagged_2.append(df)

Compare batch 1's tagged v1 shape versus v2 shape

In [ ]:
# Tagged items data batch 1 v1 
print(tagged_data[0].shape)
# Tagged items data batch 1 v2
print(tagged_data[1].shape)
# Tagged items data batch 2
print(tagged_data[2].shape)

Compare all values within tagged v1 and v2

In [ ]:
# Tagged items data batch 1 v1 
tagged_data_v1 = tagged_data[0].copy()
# Tagged items data batch 1 v2
tagged_data_v2 = tagged_data[1].copy()
# Tagged items data batch 2
tagged_data_batch2 = tagged_data[2].copy()
tagged_data_v2_resized = tagged_data_v2.iloc[:tagged_data[0].shape[0], :]
changes = (tagged_data_v1.map(lambda s: s.lower().replace('.', '')) != tagged_data_v2_resized.map(lambda s: s.lower().replace('.', ''))).apply(lambda r: r.any(), axis=1)
changes.sum()

Standardize number of columns

In [ ]:
tagged_data_v2['item_description'] = "nan"
tagged_data_v2 = tagged_data_v2[tagged_data[2].columns.tolist()]
tagged_data_v2['batch'] = 1
tagged_data_batch2['batch'] = 2

Merge

In [ ]:
items_tagged = pd.concat([tagged_data_v2, tagged_data_batch2])
items_tagged.reset_index(drop=True, inplace=True)

static_reference_data_unmerged_without_tagged = []
for df1, df2 in zip(static_reference_data_unmerged_without_tagged_1, static_reference_data_unmerged_without_tagged_2):
    print(df1.columns.name)
    print(df1.shape)
    print(df2.columns.name)
    print(df2.shape)
    static_reference_data_unmerged_without_tagged.append(pd.concat([df1, df2]))

Reindex after merging

In [ ]:
before_after_details = static_reference_data_unmerged_without_tagged[0].copy()
before_after_details.reset_index(drop=True, inplace=True)
customers = static_reference_data_unmerged_without_tagged[1].copy()
customers.reset_index(drop=True, inplace=True)
locations = static_reference_data_unmerged_without_tagged[2].copy()
locations.reset_index(drop=True, inplace=True)

Format data types and entries

In [ ]:
before_after_details['cross_over_date'] = pd.to_datetime(before_after_details['cross_over_date'])
customers['age'] = customers['age'].astype('float')
items_tagged['is_plant_based'] = items_tagged['is_plant_based'].str.lower().str.replace('.', '')
items_tagged['is_plant_based'].value_counts()
items_tagged['item_name'] = items_tagged['item_name'].str.strip()
for column in locations.columns[7:-1]:
    locations[column] = locations[column].astype('float')

Format categories within attributes further

In [ ]:
items_tagged['item_type'] = items_tagged['item_type'].str.lower()
reduced_dish_categories = set((items_tagged['dish_category'].value_counts()[items_tagged['dish_category'].value_counts() > 2]).index.tolist())
items_tagged.loc[items_tagged['dish_category'].apply(lambda s: s not in reduced_dish_categories),'dish_category'] = 'Unsure'

Readd to common list of static data

In [ ]:
static_reference_data = [before_after_details, customers, items_tagged, locations]

### Restaurant Sales Data

Read sales data for each restaurant

In [ ]:
restaurant_sales_data = {}
# Read restaurant dataframes
for batch_index in [1,2]:
    location_filenames = os.listdir(f"palate_data_parquet/batch_{batch_index}/orders_item_level")
    for location_filename in tqdm(location_filenames):
        location_id = re.sub(r'\.parquet$', '', location_filename)
        df = pd.read_parquet(f"palate_data_parquet/batch_{batch_index}/orders_item_level/" + location_filename)

        df.set_index(keys="created_at", drop=True, inplace=True)
        df.sort_index(inplace=True)
        
        restaurant_sales_data[location_id] = df

Merge part 1 and part 2 of largest restaurant file

In [ ]:
large_file_location_id = '2HRX9P6HKXA8V'
restaurant_sales_data[large_file_location_id] = pd.concat([restaurant_sales_data[f'{large_file_location_id}_part1'], restaurant_sales_data[f'{large_file_location_id}_part2']])
del restaurant_sales_data[f'{large_file_location_id}_part1']
del restaurant_sales_data[f'{large_file_location_id}_part2']

Formatting entries

In [ ]:
for location_id in restaurant_sales_data.keys():
    restaurant_sales_data[location_id]['item_name'] = restaurant_sales_data[location_id]['item_name'].str.strip()

New price per unit column

In [ ]:
# Determine a per unit price since prices are per order
for location_id in restaurant_sales_data.keys():
    restaurant_sales_data[location_id]['unit_price'] = restaurant_sales_data[location_id]['item_price'] / restaurant_sales_data[location_id]['item_quantity']

Store lists of dataframes

In [ ]:
%store static_reference_data
%store restaurant_sales_data